# Publish Qwen3.5-4B from Kaggle into our private dataset

This notebook runs **on Kaggle**, never on the laptop: Kaggle downloads the 9.4 GB snapshot to its
own disk and publishes the private dataset `rtp-model-qwen35-4b-851bf6e8`, which our notebooks then
attach read-only at `/kaggle/input`. The laptop never uploads the model.

The only secret needed is `KAGGLE_API_TOKEN`, attached to this notebook in Kaggle's Secrets panel.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

MODEL = "Qwen/Qwen3.5-4B"
REVISION = "851bf6e8"
DATASET_SLUG = "rtp-model-qwen35-4b-851bf6e8"      # == assetcache slug for this spec
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
DEST = WORK / DATASET_SLUG


def say(*a):
    print(*a, flush=True)


def creds() -> None:
    if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME")
                                              and os.environ.get("KAGGLE_KEY")):
        say("kaggle credentials: present in the environment")
        return
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception as exc:
        raise SystemExit(f"no Kaggle credentials and no secrets client ({type(exc).__name__})")
    client = UserSecretsClient()
    # A 400 here usually means the secret was added to the account but not attached to THIS
    # notebook (Settings -> Secrets -> toggle this notebook), or the label differs in case.
    for label in ("KAGGLE_API_TOKEN", "KAGGLE_TOKEN", "kaggle_api_token", "KAGGLE_KEY"):
        try:
            value = client.get_secret(label)
        except Exception as exc:                              # noqa: BLE001
            say(f"secret {label}: unavailable ({type(exc).__name__})")
            continue
        if value:
            os.environ["KAGGLE_API_TOKEN"] = value
            say(f"kaggle credentials: read from Kaggle Secrets ({label})")
            return
    raise SystemExit("no Kaggle credentials: attach a secret named KAGGLE_API_TOKEN to THIS notebook "
                     "in the Kaggle editor (Secrets -> Add a secret -> toggle this notebook)")


def download() -> None:
    from huggingface_hub import snapshot_download
    DEST.mkdir(parents=True, exist_ok=True)
    say(f"downloading {MODEL}@{REVISION} -> {DEST}")
    snapshot_download(MODEL, revision=REVISION, local_dir=str(DEST),
                      token=os.environ.get("HF_TOKEN"))
    # The hub keeps bookkeeping under <local_dir>/.cache; a symlinked blob dir would break the
    # upload if we deleted it, so materialise any symlinks first.
    cache = DEST / ".cache"
    links = [p for p in DEST.rglob("*") if p.is_symlink()]
    for link in links:
        real = link.resolve()
        tmp = link.with_suffix(link.suffix + ".real")
        shutil.copy2(real, tmp)
        link.unlink()
        tmp.rename(link)
    if cache.exists():
        shutil.rmtree(cache, ignore_errors=True)
    total = sum(p.stat().st_size for p in DEST.rglob("*") if p.is_file())
    say(f"dataset files: {sum(1 for p in DEST.rglob('*') if p.is_file())}, {total / 1e9:.2f} GB, "
        f"symlinks materialised: {len(links)}")


def write_metadata() -> None:
    (DEST / "dataset-metadata.json").write_text(json.dumps({
        "title": "rtp model qwen35 4b 851bf6e8",
        "id": f"{os.environ.get('KAGGLE_USERNAME') or _username()}/{DATASET_SLUG}",
        "licenses": [{"name": "other"}],
        "isPrivate": True,
        "description": (f"Private snapshot of {MODEL} at revision {REVISION}, published from Kaggle "
                        "so the laptop never uploads 9.4 GB. Consumed by "
                        "rtp.remotes.RemoteStore.resolve."),
    }, indent=2))
    say("wrote dataset-metadata.json (isPrivate=true)")


def _username() -> str:
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        return api.config_values["username"]
    except Exception as exc:                                  # noqa: BLE001
        raise SystemExit(f"cannot determine the Kaggle username ({type(exc).__name__})")


def publish() -> None:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    owner = api.config_values["username"]
    os.environ.setdefault("KAGGLE_USERNAME", owner)
    try:
        api.dataset_status(DATASET_SLUG)
        exists = True
    except Exception:                                        # noqa: BLE001
        exists = False
    say(f"dataset {owner}/{DATASET_SLUG}: {'exists, adding a version' if exists else 'creating'}")
    if exists:
        api.dataset_create_version(folder=str(DEST), version_notes=f"{MODEL}@{REVISION}",
                                   quiet=True, convert_to_csv=False, dir_mode="skip")
    else:
        api.dataset_create_new(folder=str(DEST), public=False, quiet=True,
                               convert_to_csv=False, dir_mode="skip")
    say("published")


def main() -> int:
    creds()
    if not (DEST / "config.json").exists():
        download()
    write_metadata()
    publish()
    subprocess.run([sys.executable, "-c", "print('done')"], check=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())